# Computes and plots the urban wind as forecasted for today.
## kernel: polytope

# User selection:
## meteo type: ECMWF or on demand DT
## size and locations of the map: user can select the center and the size of the maps (EPSG: 31370)


In [ ]:
from polytope.api import Client
import numpy as np
import pandas as pd
from src.urban_wind import get_wind_extreme_dt, read_cfd_wind, scale_cfd_wind, save_local_wind, get_wind_extreme_dt_ondemand
from src.green_pont import load_json_with_comments
from src.plots_jupyter import wind_animation
from IPython.display import HTML

from matplotlib import animation
import matplotlib.pyplot as plt

LIVE_REQUEST = True
cf=load_json_with_comments('etc/settings_wind_maps.json')
path_cfd=cf['path_cfd'] 
angles=cf["angles"]
height=cf["height"]

# Request data from Extreme dt
LOCATION = ((51.213642, 4.415)) #  center of antwerp to get meteo data from ecmwf

In [ ]:
meteo_request="on_demand" # optios are: "on_demand" or "ECMWF"

# read pre-computed normalized CFD wind ratios
xc=153300 #center of antwerp
yc=211644 #center of antwerp
L=1000    # size of the map that will be created: if too large it insula crashes

In [ ]:
crop_bounds = (xc-L, yc-L, xc+L, yc+L)
cfd_ratio=read_cfd_wind(path_cfd,angles, height,crop_bounds)

In [ ]:
cfd_ratio['x']

In [ ]:
if meteo_request=="ECMWF":
    wind_meteo=get_wind_extreme_dt(LOCATION,date='0')
elif meteo_request=="on_demand": 
    wind_meteo=get_wind_extreme_dt_ondemand(LOCATION,date="2023-08-20")
else:
    print('Unknown meteo request')

In [ ]:
# scale meso-scale wind to local urban scale using CFD ratios
wind_local=scale_cfd_wind(wind_meteo, cfd_ratio)
print('Done processing local wind')

In [ ]:
times = sorted(wind_local.keys())

fig, ax = plt.subplots(figsize=(8,8))
all_values = np.concatenate([wind_local[t].ravel() for t in times])
vmin, vmax = np.percentile(all_values, [2, 98])  # robust color scale

im = ax.imshow(wind_local[times[0]], cmap='jet', animated=True, vmin=0, vmax=3)
cbar = fig.colorbar(im, ax=ax)
title = ax.set_title(f"Wind velocity [m/s] at {times[0]}")
ax.axis("off")

# Update function for animation
def update(frame):
    data = wind_local[times[frame]]
    im.set_array(data)
    title.set_text(f"Wind field at {times[frame]}")
    return [im, title]

# Build the animation
ani = animation.FuncAnimation(
    fig, update, frames=len(times), interval=800, blit=True
)

# Show in Jupyter
HTML(ani.to_jshtml())

In [ ]:
plt.close()

In [ ]:
save_local_wind(wind_local, cfd_ratio, cf["output_path"])